# 07_simulate_top3_policy.ipynb

Notebook này mô phỏng lớp quyết định phản hồi **sau retrieval, trước chatbot**.

Mục tiêu:
- dùng **top-3** thay vì phụ thuộc cứng vào top-1
- phân loại kết quả retrieval thành các mode phản hồi:
  - `informational_test`
  - `emergency_or_urgent`
  - `mixed_emergency`
  - `fallback`
- thử nghiệm policy trên retriever v1 hiện tại


In [1]:
from pathlib import Path
import json
from collections import Counter
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())
ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / "retriever_v1"

KB_CHUNKS_PATH = ARTIFACTS_DIR / "kb_chunks_v1.json"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "faiss.index"
EMBEDDING_CONFIG_PATH = ARTIFACTS_DIR / "embedding_config.json"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("KB_CHUNKS_PATH =", KB_CHUNKS_PATH)
print("FAISS_INDEX_PATH =", FAISS_INDEX_PATH)
print("EMBEDDING_CONFIG_PATH =", EMBEDDING_CONFIG_PATH)


AI_LAB_ROOT = D:\homelab\ai_lab
KB_CHUNKS_PATH = D:\homelab\ai_lab\artifacts\retriever_v1\kb_chunks_v1.json
FAISS_INDEX_PATH = D:\homelab\ai_lab\artifacts\retriever_v1\faiss.index
EMBEDDING_CONFIG_PATH = D:\homelab\ai_lab\artifacts\retriever_v1\embedding_config.json


In [2]:
with open(KB_CHUNKS_PATH, "r", encoding="utf-8") as f:
    kb_chunks = json.load(f)

with open(EMBEDDING_CONFIG_PATH, "r", encoding="utf-8") as f:
    embedding_config = json.load(f)

index = faiss.read_index(str(FAISS_INDEX_PATH))
model = SentenceTransformer(embedding_config["model_name"])

print("Chunks:", len(kb_chunks))
print("Index total:", index.ntotal)
print("Model:", embedding_config["model_name"])
pd.DataFrame(kb_chunks)[["chunk_id", "title", "section", "source_id", "risk_level", "faq_type"]].head(15)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks: 15
Index total: 15
Model: intfloat/multilingual-e5-small


,chunk_id,title,section,source_id,risk_level,faq_type
0,kb_v1_001_c1,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,nice_sepsis_overview,high,red_flag_general
1,kb_v1_002_c1,Một số dấu hiệu bên ngoài làm tăng mức độ cảnh...,red_flags,nice_sepsis_overview,high,red_flag_signs
2,kb_v1_003_c1,Một số dấu hiệu có thể làm mức độ lo ngại tăng...,red_flags,nice_sepsis_overview,high,red_flag_signs
3,kb_v1_004_c1,Khi có dấu hiệu nguy cơ cao của nhiễm trùng nặ...,red_flags,nice_sepsis_overview,high,emergency_warning
4,kb_v1_005_c1,Xét nghiệm máu là gì,test_explainers,blood_tests,low,general_info
5,kb_v1_006_c1,Vì sao bác sĩ có thể chỉ định xét nghiệm máu,test_explainers,blood_tests,low,general_info
6,kb_v1_007_c1,Một số loại xét nghiệm máu thường gặp,test_explainers,blood_tests,low,test_meaning
7,kb_v1_008_c1,Cần chuẩn bị gì trước khi xét nghiệm máu,test_explainers,blood_tests,low,preparation
8,kb_v1_009_c1,Sau khi xét nghiệm máu và khi nhận kết quả,test_explainers,blood_tests,low,results
9,kb_v1_010_c1,Khi nào đau ngực cần gọi cấp cứu ngay,red_flags,chest_pain,high,emergency_warning


## Search helper

Hàm `search()` trả về top-k chunk cùng score để policy lớp trên ra quyết định.


In [3]:
def search(query, top_k=3):
    query_text = embedding_config.get("query_prefix", "query: ") + query.strip()
    q_emb = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = kb_chunks[idx]
        results.append({
            "rank": len(results) + 1,
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "kb_id": chunk["kb_id"],
            "source_id": chunk["source_id"],
            "source_name": chunk["source_name"],
            "section": chunk["section"],
            "title": chunk["title"],
            "faq_type": chunk.get("faq_type", ""),
            "risk_level": chunk.get("risk_level", "unknown"),
            "content_preview": chunk["content"][:400],
            "safety_notes": chunk.get("safety_notes", "")
        })
    return results


## Response policy v1

Bám theo policy đã chốt:
- `informational_test`
- `emergency_or_urgent`
- `mixed_emergency`
- `fallback`


In [4]:
EMERGENCY_SOURCES = {"chest_pain", "shortness_of_breath", "nice_sepsis_overview"}

def classify_top3(results):
    if not results:
        return {
            "mode": "fallback",
            "reason": "no_results",
            "summary": "Không có kết quả retrieval."
        }

    top1 = results[0]
    sections = [r["section"] for r in results]
    sources = [r["source_id"] for r in results]

    section_counts = Counter(sections)
    source_counts = Counter(sources)

    emergency_hits = [
        r for r in results
        if r["section"] == "red_flags" and r["source_id"] in EMERGENCY_SOURCES
    ]
    emergency_source_count = len(set(r["source_id"] for r in emergency_hits))

    # Rule 1: mixed emergency nếu top-3 có >=2 source red_flags khác nhau
    if emergency_source_count >= 2:
        return {
            "mode": "mixed_emergency",
            "reason": "multiple_red_flag_sources_in_top3",
            "summary": "Top-3 chứa nhiều nguồn cảnh báo nguy hiểm chồng lấp.",
            "source_counts": dict(source_counts),
            "section_counts": dict(section_counts)
        }

    # Rule 2: top-1 là red_flags => emergency_or_urgent
    if top1["section"] == "red_flags":
        return {
            "mode": "emergency_or_urgent",
            "reason": "top1_is_red_flag",
            "summary": "Top-1 là chunk red_flags nên cần phản hồi ưu tiên an toàn.",
            "source_counts": dict(source_counts),
            "section_counts": dict(section_counts)
        }

    # Rule 3: top-1 là blood_tests và top-3 không có red_flags
    if top1["source_id"] == "blood_tests" and "red_flags" not in sections:
        return {
            "mode": "informational_test",
            "reason": "blood_tests_without_red_flags",
            "summary": "Top-3 nghiêng rõ về giải thích xét nghiệm.",
            "source_counts": dict(source_counts),
            "section_counts": dict(section_counts)
        }

    return {
        "mode": "fallback",
        "reason": "unclear_top3_pattern",
        "summary": "Top-3 chưa đủ rõ để chọn mode an toàn.",
        "source_counts": dict(source_counts),
        "section_counts": dict(section_counts)
    }


## Build demo answer

Đây là câu trả lời mô phỏng để bạn xem policy chọn mode nào.
Chưa phải câu trả lời backend cuối cùng.


In [5]:
def build_demo_answer(results):
    decision = classify_top3(results)

    if not results:
        return {
            "mode": "fallback",
            "decision": decision,
            "answer": "Mình chưa tìm được thông tin phù hợp trong kho tri thức hiện tại."
        }

    top1 = results[0]

    if decision["mode"] == "informational_test":
        answer = top1["content_preview"]
        return {
            "mode": decision["mode"],
            "decision": decision,
            "answer": answer
        }

    if decision["mode"] == "emergency_or_urgent":
        answer = f"{top1['content_preview']} {top1.get('safety_notes', '')}".strip()
        return {
            "mode": decision["mode"],
            "decision": decision,
            "answer": answer
        }

    if decision["mode"] == "mixed_emergency":
        answer = (
            "Các dấu hiệu bạn mô tả đang chồng lấp nhiều nhóm cảnh báo nguy hiểm, "
            "ví dụ đau ngực, khó thở hoặc dấu hiệu nhiễm trùng nặng. "
            "Bạn nên gọi cấp cứu hoặc đến cơ sở y tế khẩn cấp ngay, "
            "thay vì tự theo dõi tại nhà."
        )
        return {
            "mode": decision["mode"],
            "decision": decision,
            "answer": answer
        }

    return {
        "mode": "fallback",
        "decision": decision,
        "answer": "Mình chưa đủ chắc chắn để trả lời. Bạn có thể mô tả cụ thể hơn triệu chứng hoặc nội dung cần hỏi không?"
    }


## Test với các query mẫu

In [6]:
test_queries = [
    "xét nghiệm máu là gì",
    "cần chuẩn bị gì trước khi xét nghiệm máu",
    "đau ngực lan ra tay và khó thở có nguy hiểm không",
    "khó thở nặng tím môi và lú lẫn có cần cấp cứu không",
    "người bệnh nhiễm trùng xấu đi nhanh, tím môi và lú lẫn"
]

for q in test_queries:
    results = search(q, top_k=3)
    response = build_demo_answer(results)

    print("=" * 100)
    print("QUERY:", q)
    print("MODE :", response["mode"])
    print("REASON:", response["decision"]["reason"])
    print("TOP-3:")
    for r in results:
        print(f"  [{r['rank']}] {r['source_id']} | {r['section']} | {r['title']} | score={r['score']:.4f}")
    print("ANSWER:")
    print(response["answer"])
    print()


QUERY: xét nghiệm máu là gì
MODE : informational_test
REASON: blood_tests_without_red_flags
TOP-3:
  [1] blood_tests | test_explainers | Xét nghiệm máu là gì | score=0.9397
  [2] blood_tests | test_explainers | Vì sao bác sĩ có thể chỉ định xét nghiệm máu | score=0.9115
  [3] blood_tests | test_explainers | Một số loại xét nghiệm máu thường gặp | score=0.9038
ANSWER:
Xét nghiệm máu là một xét nghiệm y khoa phổ biến, thường được dùng để kiểm tra sức khỏe tổng quát hoặc hỗ trợ tìm nguyên nhân của một số triệu chứng. Khi làm xét nghiệm, nhân viên y tế sẽ lấy một lượng máu nhỏ để gửi đến phòng xét nghiệm.

QUERY: cần chuẩn bị gì trước khi xét nghiệm máu
MODE : informational_test
REASON: blood_tests_without_red_flags
TOP-3:
  [1] blood_tests | test_explainers | Cần chuẩn bị gì trước khi xét nghiệm máu | score=0.9511
  [2] blood_tests | test_explainers | Vì sao bác sĩ có thể chỉ định xét nghiệm máu | score=0.9030
  [3] blood_tests | test_explainers | Một số loại xét nghiệm máu thường gặp | s

## Test một query thủ công

In [7]:
# Thay câu hỏi ở đây để test thủ công
query = "đau ngực kèm khó thở và vã mồ hôi có cần gọi cấp cứu không"

results = search(query, top_k=3)
response = build_demo_answer(results)

print("QUERY:", query)
print("MODE :", response["mode"])
print("REASON:", response["decision"]["reason"])
print("\nTOP-3 RESULTS:")
for r in results:
    print(f"[{r['rank']}] score={r['score']:.4f} | {r['source_id']} | {r['section']} | {r['title']}")

print("\nANSWER:")
print(response["answer"])


QUERY: đau ngực kèm khó thở và vã mồ hôi có cần gọi cấp cứu không
MODE : mixed_emergency
REASON: multiple_red_flag_sources_in_top3

TOP-3 RESULTS:
[1] score=0.9302 | chest_pain | red_flags | Khi nào đau ngực cần gọi cấp cứu ngay
[2] score=0.9010 | shortness_of_breath | red_flags | Khi nào khó thở cần hỗ trợ khẩn cấp
[3] score=0.8763 | shortness_of_breath | red_flags | Khi nào khó thở cần được đánh giá y tế sớm

ANSWER:
Các dấu hiệu bạn mô tả đang chồng lấp nhiều nhóm cảnh báo nguy hiểm, ví dụ đau ngực, khó thở hoặc dấu hiệu nhiễm trùng nặng. Bạn nên gọi cấp cứu hoặc đến cơ sở y tế khẩn cấp ngay, thay vì tự theo dõi tại nhà.


## Xuất kết quả test ra DataFrame

In [8]:
rows = []
for q in test_queries:
    results = search(q, top_k=3)
    response = build_demo_answer(results)
    for r in results:
        rows.append({
            "query": q,
            "mode": response["mode"],
            "rank": r["rank"],
            "score": r["score"],
            "source_id": r["source_id"],
            "section": r["section"],
            "title": r["title"]
        })

df_policy = pd.DataFrame(rows)
df_policy


,query,mode,rank,score,source_id,section,title
0,xét nghiệm máu là gì,informational_test,1,0.939669,blood_tests,test_explainers,Xét nghiệm máu là gì
1,xét nghiệm máu là gì,informational_test,2,0.911544,blood_tests,test_explainers,Vì sao bác sĩ có thể chỉ định xét nghiệm máu
2,xét nghiệm máu là gì,informational_test,3,0.903754,blood_tests,test_explainers,Một số loại xét nghiệm máu thường gặp
3,cần chuẩn bị gì trước khi xét nghiệm máu,informational_test,1,0.951128,blood_tests,test_explainers,Cần chuẩn bị gì trước khi xét nghiệm máu
4,cần chuẩn bị gì trước khi xét nghiệm máu,informational_test,2,0.903046,blood_tests,test_explainers,Vì sao bác sĩ có thể chỉ định xét nghiệm máu
5,cần chuẩn bị gì trước khi xét nghiệm máu,informational_test,3,0.900133,blood_tests,test_explainers,Một số loại xét nghiệm máu thường gặp
6,đau ngực lan ra tay và khó thở có nguy hiểm không,mixed_emergency,1,0.889222,shortness_of_breath,red_flags,Khi nào khó thở cần hỗ trợ khẩn cấp
7,đau ngực lan ra tay và khó thở có nguy hiểm không,mixed_emergency,2,0.886005,shortness_of_breath,red_flags,Khi nào khó thở cần được đánh giá y tế sớm
8,đau ngực lan ra tay và khó thở có nguy hiểm không,mixed_emergency,3,0.882488,chest_pain,red_flags,Khi nào đau ngực cần gọi cấp cứu ngay
9,khó thở nặng tím môi và lú lẫn có cần cấp cứu ...,mixed_emergency,1,0.909993,shortness_of_breath,red_flags,Khi nào khó thở cần hỗ trợ khẩn cấp


## Gợi ý đọc kết quả

- Nếu query về xét nghiệm máu → mode nên là `informational_test`
- Nếu query có đau ngực, khó thở, sepsis red flags → mode nên là `emergency_or_urgent` hoặc `mixed_emergency`
- Nếu top-3 có nhiều nguồn red_flags khác nhau → ưu tiên `mixed_emergency`
